# Phase 6: Abstractive Legal Summarisation with the pre-trained **Transformer-XL + GQA** (decoder-only)

This is the Transformer-XL twin of the GQLA Phase-6 notebook. Data, the memory-bridge design, beam search, metrics, PEFT modules, the training loop,
KD and submission are the same, so results compare directly. Only the backbone code changes: the model class comes from your
Transformer-XL Phase-4/7 notebooks, and a new attention forward adds a KV cache for generation.

Transformer-XL has no encoder. As in the GQLA notebook, the judgment is **compressed into a short memory** that the model reads as soft tokens:

```
document → chunks <s> + 510 tokens + </s>  (K spread over the judgment)
        → frozen Phase-1 Transformer-XL, run once     → hidden states of each chunk
        → m segment-means per chunk                    → K·m memory slots (cached fp16 on Drive)
        → MemoryBridge (trainable): chunk/slot positions + 1 bidirectional Transformer layer + projection to the embedding space
        → Transformer-XL (adapted with the chosen method) reads ONE causal sequence of ≤ 512 positions:

          [ prompt? | pad … pad | mem_1 … mem_N | <s> | y_1 … y_L | </s> ]
                                                 └── loss only here (label smoothing 0.1)
```

**What is specific to Transformer-XL**

| | GQLA notebook | This notebook |
|---|---|---|
| attention | GQLA + decoupled RoPE, fused SDPA | Transformer-XL relative attention `(q+u)·k + shift((q+v)·r)`. `(q+u)·k` runs inside SDPA and the positional term is the additive mask (same lean form as your Phase-7 notebook) |
| KV cache | grouped `k_C, v_C` + un-rotated RoPE key | grouped `k`, `v` (8 KV heads). Relative positions are recomputed each step from the key length, so there is nothing positional to store |
| left padding | key-padding mask | key-padding mask. Relative attention depends only on distances, so left padding leaves the real positions unchanged (self-tested) |
| prefix tuning | learned prefix pushed through `W_DKV/W_UK/W_UV` | learned **Transformer-XL memory** per block (its own `memory` slot), initialised from real layer inputs |
| LoRA / LoRMA⁺ / QLoRA targets | `W_DQ … W_O`, FFN | `q_proj, k_proj, v_proj, out_proj, ffn.0, ffn.3` (`r_proj`, `u`, `v` stay frozen) |
| KD student | 5 of 10 layers | **4 of 8 blocks, layer-strided** `[0, 2, 5, 7]`, initialised from the fine-tuned teacher (as in your TXL Phase-3/4/7) |
| segment recurrence | n/a | not used across chunks (each chunk is encoded independently). Say so in the report |

**Methods** (`method=`): `full` · `partial` · `lora` · `lorma` (LoRMA⁺, compulsory) · `qlora` · `houlsby` · `pfeiffer` · `prefix` · `prompt` · `frozen` (bridge only) · `kd`.
`mem_mode="lead"` is the plain prefix-LM baseline, where the first document tokens are fed as text.

**Run order:** top to bottom. Cell *1b* self-tests the new attention against the Phase-1 attention (with and without memory), cached decoding against a full forward,
left padding, and fp16. The config cell checks that every PEFT method is the identity at initialisation. Then run the smoke test, then the experiments.

**Limit:** the summary has to fit in what the memory leaves of the 512 window (about 380 tokens with 128 memory slots). The config cell prints how many references are longer than that.
All hyper-parameters are starting points, not tuned values (course rules §17.5).

In [ ]:
# metrics used for local validation (the official scorer computes BERTScore + ROUGE the same way)
!pip -q install rouge-score bert-score

In [1]:
# ============================================================
# 0. SETUP
# ============================================================
import os, sys, re, json, math, time, random, copy, pickle, shutil, gc, glob, unicodedata
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # fp16 on T4
print("mixed precision:", AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print("HF token set:", bool(HF_TOKEN))

# ---- Phase-1 Transformer-XL + GQA artefacts (same paths as your TXL Phase-4/7 notebooks; edit if yours differ) ----
P1_DIR = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
CHECKPOINT_PATH = os.path.join(P1_DIR, "transformer_xl_gqa_checkpoints", "transformer_xl_gqa-large.pt")
TOKENIZER_PATH = os.path.join(P1_DIR, "final_bpe_en.json")
MODEL_CONFIG = {"vocab_size": 32768, "d_model": 512, "num_layers": 32, "num_query_heads": 64, "num_kv_heads": 16,
                "memory_length": 128, "dropout": 0.1}
TXL_CTX_LEN = 512                      # window used in Phase 1 / 4 / 7: the summariser's whole sequence stays within it

p6_dir   = "/content/drive/MyDrive/Phase6_Summ_TXL"; os.makedirs(p6_dir, exist_ok=True)
local_ck = "/content/p6_ckpt_txl"; os.makedirs(local_ck, exist_ok=True)      # fast local disk; best ckpts copied to Drive
FEAT_LOCAL = "/content/feat_cache_txl6"; os.makedirs(FEAT_LOCAL, exist_ok=True)
for what, p in (("checkpoint", CHECKPOINT_PATH), ("tokenizer", TOKENIZER_PATH)):
    if not os.path.isfile(p):
        raise FileNotFoundError(f"Phase-1 {what} file not found (or it is a folder): {p}")

Mounted at /content/drive
device: cuda | Tesla T4
mixed precision: torch.float16
HF token set: True


## Phase-1 tokenizer + text normalisation (identical to the TXL Phase-4/7 notebooks)

In [2]:
import os, re, json, heapq, collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:
    _PROC = None

def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0

WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)
_WORKER = None
def _init_worker(tok):
    global _WORKER
    _WORKER = tok
def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]

class BPETokenizer:
    _HEAP_THRESHOLD = 24
    _CACHE_LIMIT = 500_000

    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = reserved_tokens if reserved_tokens is not None else ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}
        self.merges = {}
        self._ready = False
        self._cache = {}

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = BPETokenizer(vocab_size=data["vocab_size"], end_marker=data.get("end_marker", "</w>"),
                            reserved_tokens=data.get("reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]))
        tok.itos = data["itos"]; tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False; tok._cache = {}
        return tok

    def _prepare_inference(self):
        if self._ready: return
        self._byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]; self._eos = self.stoi["</s>"]
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError("Merge/vocab mismatch: cannot infer merge ids from ranks.")
        self._M = len(self.itos); M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}
        self._cache = {}; self._ready = True

    def _merge_small(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i
            if best is None: break
            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return ids

    def _merge_heap(self, ids):
        get, M, base = self._ranks.get, self._M, self._first_merge_id
        n = len(ids)
        nxt = list(range(1, n + 1)); nxt[-1] = -1
        prv = list(range(-1, n - 1)); alive = [True] * n
        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None: heap.append((r, i))
        heapq.heapify(heap)
        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]: continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r: continue
            ids[i] = base + r; alive[j] = False
            k = nxt[j]; nxt[i] = k
            if k != -1: prv[k] = i
            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None: heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None: heapq.heappush(heap, (r2, i))
        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None: return hit
        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)
        ids = self._merge_small(ids) if len(ids) <= self._HEAP_THRESHOLD else self._merge_heap(ids)
        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT: self._cache[word] = result
        return result

    def encode(self, text, add_special_tokens=True):
        self._prepare_inference()
        out = [self._bos] if add_special_tokens else []
        extend = out.extend; cache_get = self._cache.get; bpe = self._bpe_word
        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None: t = bpe(w)
            extend(t)
        if add_special_tokens: out.append(self._eos)
        return out

    def encode_batch(self, texts, add_special_tokens=True, num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)
        if num_workers is None: num_workers = os.cpu_count() or 1
        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]
        import multiprocessing as mp
        chunks = [(texts[i:i + chunk_size], add_special_tokens) for i in range(0, len(texts), chunk_size)]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker, initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        self._prepare_inference()
        filtered_ids = [tid for tid in token_ids if self.itos[tid] not in self.reserved_tokens]
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

if not os.path.isfile(TOKENIZER_PATH): raise FileNotFoundError(TOKENIZER_PATH)
tokenizer = BPETokenizer.from_json(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get("<pad>", 0); BOS_ID = tokenizer.stoi.get("<s>", 2); EOS_ID = tokenizer.stoi.get("</s>", 3)
VOCAB_SIZE = len(tokenizer.itos)
assert VOCAB_SIZE == MODEL_CONFIG["vocab_size"], "tokenizer vocab must match the Phase-1 checkpoint"
print("vocab", VOCAB_SIZE, "| PAD/BOS/EOS", PAD_ID, BOS_ID, EOS_ID)


def normalize_text(x):
    """the cleaning used by the Phase-4 Transformer-XL notebook (NFKC + whitespace collapse)"""
    s = unicodedata.normalize("NFKC", "" if x is None else str(x))
    return re.sub(r"\s+", " ", s.replace("\u00a0", " ")).strip()

vocab 32768 | PAD/BOS/EOS 0 2 3


## Phase-1 Transformer-XL + GQA (classes copied unchanged; the checkpoint must load with no missing/unexpected keys)

In [3]:
import math

class TransformerXLRelativePositionalEncoding(nn.Module):
    def __init__(self, d_model: int, base: float = 10000.0):
        super().__init__()
        if d_model % 2:
            raise ValueError("d_model must be even")
        self.d_model = d_model
        inv_freq = 1.0 / (base ** (torch.arange(0, d_model, 2, dtype=torch.float32) / d_model))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, query_length: int, key_length: int, device=None):
        if key_length <= 0:
            raise ValueError("key_length must be positive")
        if device is None:
            device = self.inv_freq.device
        pos_seq = torch.arange(key_length - 1, -1, -1, dtype=torch.float32, device=device)
        sinusoid_inp = torch.outer(pos_seq, self.inv_freq.to(device))
        return torch.cat([sinusoid_inp.sin(), sinusoid_inp.cos()], dim=-1)

class TransformerXLGQAAttention(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0):
        super().__init__()
        if d_model % num_query_heads != 0:
            raise ValueError("d_model must be divisible by num_query_heads")
        if num_query_heads % num_kv_heads != 0:
            raise ValueError("num_query_heads must be divisible by num_kv_heads")
        self.d_model = d_model
        self.num_query_heads = num_query_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_query_heads
        self.group_size = num_query_heads // num_kv_heads

        self.q_proj = nn.Linear(d_model, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.r_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.u = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        self.v = nn.Parameter(torch.zeros(num_query_heads, self.head_dim))
        nn.init.normal_(self.u, std=0.02)
        nn.init.normal_(self.v, std=0.02)
        self.dropout = nn.Dropout(dropout)

    def _q_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_query_heads, self.head_dim).transpose(1, 2)

    def _kv_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

    def _expand_kv(self, x):
        B, H, T, D = x.shape
        return x.unsqueeze(2).expand(B, H, self.group_size, T, D).reshape(B, self.num_query_heads, T, D)

    @staticmethod
    def _rel_shift(x):
        B, H, Q, K = x.shape
        zero = torch.zeros((B, H, Q, 1), dtype=x.dtype, device=x.device)
        x = torch.cat([zero, x], dim=-1)
        x = x.view(B, H, K + 1, Q)
        x = x[:, :, 1:].view(B, H, Q, K)
        return x

    @staticmethod
    def _causal_mask(query_length, key_length, device):
        memory_length = key_length - query_length
        return torch.triu(torch.ones((query_length, key_length), dtype=torch.bool, device=device),
                           diagonal=memory_length + 1)

    def forward(self, x, memory=None, relative_position=None):
        B, Q, _ = x.shape
        if memory is not None and memory.size(1) > 0:
            kv_input = torch.cat([memory, x], dim=1)
        else:
            kv_input = x
        K = kv_input.size(1)

        q = self._q_heads(self.q_proj(x))
        k = self._expand_kv(self._kv_heads(self.k_proj(kv_input)))
        v = self._expand_kv(self._kv_heads(self.v_proj(kv_input)))

        ac = torch.einsum("bhqd, bhkd->bhqk", q, k)
        ac = ac + torch.einsum("hd,bhkd->bhk", self.u, k).unsqueeze(2)

        if relative_position is None:
            raise ValueError("relative_position is required for Transformer-XL attention")

        r = self._expand_kv(self._kv_heads(self.r_proj(relative_position).unsqueeze(0)))
        bd = torch.einsum("bhqd,bhkd->bhqk", q, r)
        bd = bd + torch.einsum("hd,bhkd->bhk", self.v, r).unsqueeze(2)
        bd = self._rel_shift(bd)

        scores = (ac + bd) / math.sqrt(self.head_dim)
        mask = self._causal_mask(Q, K, x.device)
        scores = scores.masked_fill(mask.view(1, 1, Q, K), torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, Q, self.d_model)
        return self.out_proj(out)


class TransformerXLBlockGQA(nn.Module):
    def __init__(self, d_model, num_query_heads, num_kv_heads, dropout=0.0, ffn_mult=4):
        super().__init__()
        self.attn = TransformerXLGQAAttention(d_model, num_query_heads, num_kv_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        ffn_dim = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, memory, relative_position):
        x = self.norm1(x + self.attn(x, memory=memory, relative_position=relative_position))
        x = self.norm2(x + self.ffn(x))
        return x


class TransformerXLGQAModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_query_heads, num_kv_heads, memory_length,
                 dropout=0.0, tie_embeddings=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.memory_length = memory_length
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = TransformerXLRelativePositionalEncoding(d_model)
        self.blocks = nn.ModuleList([
            TransformerXLBlockGQA(d_model, num_query_heads, num_kv_heads, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_embeddings:
            self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids, memory=None, labels=None):
        x = self.embedding(input_ids)
        Q = x.size(1)
        if memory is None:
            memory = [None] * self.num_layers
        if len(memory) != self.num_layers:
            raise ValueError(f"memory must contain {self.num_layers} entries")
        mem_len = 0 if memory[0] is None else memory[0].size(1)
        K = mem_len + Q
        rel_pos = self.pos_encoding(Q, K, device=x.device)
        for i, block in enumerate(self.blocks):
            x = block(x, memory[i], rel_pos)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size), labels.reshape(-1))
        return {"logits": logits, "loss": loss}


if not os.path.isfile(CHECKPOINT_PATH): raise FileNotFoundError(CHECKPOINT_PATH)
ck = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = ck.get("model_state_dict", ck.get("state_dict", ck)) if isinstance(ck, dict) else ck
if isinstance(ck, dict): print("checkpoint keys:", list(ck.keys())[:12], "| epoch", ck.get("epoch"), "| best_val_loss", ck.get("best_val_loss"))
txl = TransformerXLGQAModel(**MODEL_CONFIG)
missing, unexpected = txl.load_state_dict(state, strict=False)
if missing or unexpected:                      # hard error on any mismatch (same rule as the Phase-3 notebook)
    print("Missing:", missing[:10], "count =", len(missing)); print("Unexpected:", unexpected[:10], "count =", len(unexpected))
    raise RuntimeError("Checkpoint is not compatible with MODEL_CONFIG / the Transformer-XL architecture above.")
del ck, state; gc.collect()
txl.eval()
TXL_BASE = txl                                  # pristine copy: experiments always deep-copy it
print(f"Loaded Phase-1 Transformer-XL + GQA: {sum(p.numel() for n, p in txl.named_parameters() if not n.startswith('lm_head'))/1e6:.1f}M params (LM head is tied to the embedding and dropped below)")

checkpoint keys: ['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'best_val_loss', 'loss_history'] | epoch 5 | best_val_loss 6.981330989583333
Loaded Phase-1 Transformer-XL + GQA: 107.1M params (LM head is tied to the embedding and dropped below)


## Data (identical to the GQLA notebook): train.json → train / held-out val, test.json

In [4]:
# ============================================================
# DATA: Phase-6 (abstractive summarisation) from the course Hugging Face repo
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-6 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
    hits = [f for f in files if f.endswith("/" + basename) and ("summ" in f.lower() or "phase_6" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 6, got {hits}. "
                           f"Set the path by hand. Phase-6-like files: {[f for f in files if 'summ' in f.lower()]}")
    return hits[0]


_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source", "input", "article", "case_text")
_SUM_KEYS = ("summary", "summaries", "abstract", "target", "reference", "headnote", "output", "gold", "label")


def _as_text(v):
    if v is None:
        return None
    if isinstance(v, list):
        v = " ".join(str(x) for x in v)
    v = str(v).strip()
    return v or None


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    keys = list(raw[0].keys())
    print(f"{filename}: {len(raw)} records | fields: {keys}")
    # document = first known doc key present, else the longest text field; summary = first known summary key present
    doc_k = next((k for k in _DOC_KEYS if k in keys), None)
    sum_k = next((k for k in _SUM_KEYS if k in keys), None)
    if doc_k is None:
        text_keys = [k for k in keys if k != "id" and isinstance(raw[0][k], (str, list))]
        doc_k = max(text_keys, key=lambda k: len(_as_text(raw[0][k]) or ""))
    print(f"  using document field '{doc_k}', summary field '{sum_k}'")
    return [dict(id=str(r.get("id", i)), doc=_as_text(r.get(doc_k)) or "",
                 summ=_as_text(r.get(sum_k)) if sum_k else None) for i, r in enumerate(raw)]


all_train_recs = read_split("train.json")
test_recs = read_split("test.json")
_n_missing = sum(r["summ"] is None for r in all_train_recs)
if _n_missing == len(all_train_recs):
    raise KeyError("no summary field found in train.json: look at the 'fields' printed above and add the "
                   "summary field's name to _SUM_KEYS")
if _n_missing:
    print(f"dropping {_n_missing} training records with an empty summary")
all_train_recs = [r for r in all_train_recs if r["summ"] is not None and r["doc"]]

# ---- no official dev split for Phase 6: hold out a validation split from train ----------------
VAL_FRAC, SPLIT_SEED = 0.05, 1234
_perm = np.random.RandomState(SPLIT_SEED).permutation(len(all_train_recs))
_n_val = max(50, int(round(VAL_FRAC * len(all_train_recs))))
val_recs = [all_train_recs[i] for i in _perm[:_n_val]]
train_recs = [all_train_recs[i] for i in _perm[_n_val:]]
print(f"split -> train {len(train_recs)} | val {len(val_recs)}  (val is held out from train.json)")

# ---- tokenise once, cache to Drive (no special tokens: added per chunk / per sequence later) ----
# NORMALIZE_TARGETS: apply the Phase-1 normalize_text to the reference summaries too. Keep True if
# normalize_text only cleans whitespace/unicode; set False if it lower-cases (the scorer compares
# against the raw references, so a lower-cased output would cost BERTScore).
NORMALIZE_TARGETS = True


def _tok_cached(texts, tag, norm=True, bs=1000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p6_dir, f"tok_txl_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) if norm else t for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=16)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


def _tok_split(recs, tag):
    docs = _tok_cached([r["doc"] for r in recs], f"{tag}_doc_s{SPLIT_SEED}")
    sums = None
    if recs[0]["summ"] is not None:
        sums = _tok_cached([r["summ"] for r in recs], f"{tag}_sum_s{SPLIT_SEED}_n{int(NORMALIZE_TARGETS)}",
                           norm=NORMALIZE_TARGETS)
    return docs, sums


tr_doc, tr_sum = _tok_split(train_recs, "train")
va_doc, va_sum = _tok_split(val_recs, "val")
te_doc, _ = _tok_split(test_recs, "test")

# ---- length statistics: these decide the generation lengths and the context layout ------------
dl = np.array([len(x) for x in tr_doc]); sl = np.array([len(x) for x in tr_sum])
sw = np.array([len(r["summ"].split()) for r in train_recs])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 95, 99))
print("document BPE tokens :", pct(dl), f" max {dl.max()}")
print("summary  BPE tokens :", pct(sl), f" max {sl.max()}")
print("summary  words      :", pct(sw))
print(f"tokens per summary word ~ {sl.sum() / max(sw.sum(), 1):.2f} | summary/document length ratio median "
      f"{np.median(sl / np.maximum(dl, 1)):.3f}")
print(f"510-token chunks per document: median {np.median(np.ceil(dl / 510)):.0f}  p90 {np.percentile(np.ceil(dl / 510), 90):.0f}")

SUM_TOK_P10, SUM_TOK_P50, SUM_TOK_P90, SUM_TOK_P95 = (int(np.percentile(sl, p)) for p in (10, 50, 90, 95))
SUM_WORDS_P50 = int(np.median(sw))

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train.json: 7030 records | fields: ['id', 'document', 'summary']
  using document field 'document', summary field 'summary'


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

test.json: 100 records | fields: ['id', 'document']
  using document field 'document', summary field 'None'
dropping 2 training records with an empty summary
split -> train 6677 | val 351  (val is held out from train.json)
document BPE tokens : p10 1575  p50 3668  p90 9124  p95 13101  p99 28218  max 162967
summary  BPE tokens : p10 342  p50 753  p90 1774  p95 2399  p99 5114  max 32546
summary  words      : p10 292  p50 638  p90 1493  p95 2007  p99 4267
tokens per summary word ~ 1.19 | summary/document length ratio median 0.220
510-token chunks per document: median 8  p90 18


## Utilities: chunking, detokenisation, post-processing, ROUGE / BERTScore, submission (identical to the GQLA notebook)

In [5]:
# ============================================================
# Model-agnostic utilities: chunking, detokenisation, post-processing, ROUGE / BERTScore, submission
# ============================================================
import re, zipfile, collections


# ---------------- long-document chunking (as in Phase 5) ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    """uniform: spread the budget over the whole judgment (facts at the start, reasoning in the middle,
    the order at the end all matter for a summary). head / tail / head_tail kept for the ablation."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


# ---------------- detokenisation + post-processing ----------------
_FIXES = [(re.compile(r"\s+([.,;:!?%)\]}'])"), r"\1"),      # 'court .'  -> 'court.'
          (re.compile(r"([(\[{])\s+"), r"\1"),               # '( a )'    -> '(a)'
          (re.compile(r"(?<=\w)\s*([-/])\s*(?=\w)"), r"\1"),  # '302 / 34' -> '302/34', 'Section - 3' -> 'Section-3'
          (re.compile(r"\s{2,}"), " ")]


def detok(ids):
    """BPE ids -> text. tokenizer.decode() puts a space after every word-final token including punctuation
    ('court . the'), so spacing is repaired here. Special tokens are dropped."""
    ids = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    s = tokenizer.decode(ids)
    for pat, rep in _FIXES:
        s = pat.sub(rep, s)
    return s.strip()


def postprocess(text, max_words=None):
    """drop exact/near-exact repeated sentences (small models loop), cap length, end on a sentence boundary"""
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    seen, out = set(), []
    for s in sents:
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if len(key) < 2 or key in seen:
            continue
        seen.add(key); out.append(s)
    text = " ".join(out)
    if max_words:
        w = text.split()
        if len(w) > max_words:
            text = " ".join(w[:max_words])
    cut = max(text.rfind(". "), text.rfind("."))
    if cut > 0.8 * len(text) and cut < len(text) - 1:     # trailing fragment after the last full stop
        text = text[:cut + 1]
    return text.strip() or "."


# ---------------- metrics ----------------
try:
    from rouge_score import rouge_scorer as _rs
    _ROUGE = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
except Exception:
    _ROUGE = None


def _toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def _f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def _rouge_n(a, b, n):
    ca = collections.Counter(zip(*[a[i:] for i in range(n)])); cb = collections.Counter(zip(*[b[i:] for i in range(n)]))
    ov = sum((ca & cb).values())
    return _f1(ov / max(sum(ca.values()), 1), ov / max(sum(cb.values()), 1))


def _rouge_l(a, b):
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    return _f1(l / len(a), l / len(b))


def rouge_scores(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        if _ROUGE is not None:
            s = _ROUGE.score(r, p)
            r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
        else:                                            # fallback (no stemming): numbers differ slightly
            a, b = _toks(p), _toks(r)
            r1 += _rouge_n(a, b, 1); r2 += _rouge_n(a, b, 2); rl += _rouge_l(a, b)
    n = max(len(preds), 1)
    return dict(rouge1=r1 / n, rouge2=r2 / n, rougeL=rl / n)


BERTSCORE_KW = dict(lang="en")      # the course scorer's settings are not published; lang="en" = roberta-large
_BS = {}


def bertscore_f1(preds, refs, batch_size=16):
    """Evaluation only (the official scorer does the same). Returns None if bert-score is unavailable."""
    try:
        from bert_score import BERTScorer
    except Exception:
        return None
    key = json.dumps(BERTSCORE_KW, sort_keys=True)
    if key not in _BS:
        _BS[key] = BERTScorer(device=str(device), **BERTSCORE_KW)
    _, _, f = _BS[key].score(preds, refs, batch_size=batch_size)
    return float(f.mean())


def evaluate_texts(preds, refs, bert=True):
    m = rouge_scores(preds, refs)
    m["bertscore"] = bertscore_f1(preds, refs) if bert else None
    m["avg_words"] = float(np.mean([len(p.split()) for p in preds]))
    m["ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    return m


def fmt_metrics(m):
    bs = "n/a" if m.get("bertscore") is None else f"{m['bertscore']:.4f}"
    return (f"BERTScore {bs} | R1 {m['rouge1']:.4f} R2 {m['rouge2']:.4f} RL {m['rougeL']:.4f} | "
            f"words {m['avg_words']:.0f} (ref {m['ref_words']:.0f})")


# ---------------- submission ----------------
def write_submission(preds_by_id, name):
    out = [dict(id=r["id"], summary=preds_by_id[r["id"]]) for r in test_recs]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one summary per test id"
    assert all(isinstance(o["summary"], str) and o["summary"].strip() for o in out), "empty summary"
    d = os.path.join(p6_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(out, open(jp, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")          # at the ROOT of the zip, as Codabench requires
    print(f"wrote {len(out)} summaries (avg {np.mean([len(o['summary'].split()) for o in out]):.0f} words) -> {zp}")
    return zp

## Generic batched beam search (model-agnostic, identical to the GQLA notebook)

In [6]:
# ============================================================
# Generic batched beam search (model-agnostic)
#
# The model side supplies three closures:
#   start()            -> (state, logits[B,V])   consume the prompt / encode the source, logits for the 1st token
#   step(state, tok)   -> (state, logits[N,V])   consume one token per hypothesis (N = B*beams)
#   reorder(state, ix) -> state                  keep hypothesis rows ix (rows never move to another source:
#                                                row r always belongs to source r // beams)
# Features: length penalty, min length, no-repeat-ngram blocking, repetition penalty, early stopping.
# num_beams=1 gives greedy decoding.
# ============================================================
@torch.no_grad()
def beam_search(start, step, reorder, num_beams=4, max_new=256, min_new=0, len_penalty=1.0,
                no_repeat_ngram=3, rep_penalty=1.0, eos_id=None, banned=()):
    eos_id = EOS_ID if eos_id is None else eos_id
    state, logits = start()
    B, V = logits.shape
    K, dev = num_beams, logits.device
    state = reorder(state, torch.arange(B, device=dev).repeat_interleave(K))
    logits = logits.repeat_interleave(K, 0)
    beam_scores = torch.zeros(B, K, device=dev)
    beam_scores[:, 1:] = -1e9                       # identical beams at t=0: keep only the first
    seqs = [[] for _ in range(B * K)]
    finished = [[] for _ in range(B)]               # (normalised score, tokens)
    done = [False] * B
    banned = [t for t in banned if t is not None and t != eos_id]

    for t in range(max_new):
        lp = torch.log_softmax(logits.float(), -1)
        if banned:
            lp[:, banned] = -float("inf")
        if t < min_new:
            lp[:, eos_id] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            for r, s in enumerate(seqs):
                if s:
                    ix = torch.tensor(sorted(set(s)), device=dev)
                    lp[r, ix] -= math.log(rep_penalty)
        n = no_repeat_ngram
        if n and t >= n - 1:
            for r, s in enumerate(seqs):
                if done[r // K]:
                    continue
                prefix = tuple(s[len(s) - n + 1:]) if n > 1 else ()
                bad = [s[i + n - 1] for i in range(len(s) - n + 1) if tuple(s[i:i + n - 1]) == prefix]
                if bad:
                    lp[r, bad] = -float("inf")

        cand = (beam_scores.view(-1, 1) + lp).view(B, K * V)
        top_s, top_i = cand.topk(2 * K, dim=1)
        top_s, top_i = top_s.tolist(), top_i.tolist()
        sel_rows, sel_tok, new_scores = [], [], []
        for b in range(B):
            chosen = []
            if not done[b]:
                for sc, ix in zip(top_s[b], top_i[b]):
                    beam, tok = divmod(ix, V)
                    if sc < -1e8:
                        break
                    if tok == eos_id:
                        finished[b].append((sc / ((t + 1) ** len_penalty), list(seqs[b * K + beam])))
                    else:
                        chosen.append((b * K + beam, tok, sc))
                    if len(chosen) == K:
                        break
                # early stop: K finished hypotheses and no live beam can beat the worst of them
                if len(finished[b]) >= K:
                    best_live = max((c[2] for c in chosen), default=-1e9) / ((t + 1) ** len_penalty)
                    if best_live <= sorted(f[0] for f in finished[b])[-K]:
                        done[b] = True
            while len(chosen) < K:                    # finished source or not enough candidates: dummy rows
                chosen.append((b * K, eos_id if done[b] else (chosen[0][1] if chosen else eos_id), -1e9))
            for row, tok, sc in chosen:
                sel_rows.append(row); sel_tok.append(tok); new_scores.append(sc)
        if all(done):
            break
        seqs = [seqs[r] + [tk] for r, tk in zip(sel_rows, sel_tok)]
        beam_scores = torch.tensor(new_scores, device=dev).view(B, K)
        rows = torch.tensor(sel_rows, device=dev)
        state = reorder(state, rows)
        state, logits = step(state, torch.tensor(sel_tok, device=dev))

    out = []
    L = len(seqs[0]) if seqs else 0
    for b in range(B):
        if not done[b]:
            for k in range(K):
                sc = float(beam_scores[b, k])
                if sc > -1e8:
                    finished[b].append((sc / (max(L, 1) ** len_penalty), seqs[b * K + k]))
        out.append(max(finished[b], key=lambda x: x[0])[1] if finished[b] else [])
    return out

## Parameter-efficient adaptation modules (identical to the GQLA notebook)

In [7]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

## Training / evaluation loop (identical to the GQLA notebook)

In [8]:
# ============================================================
# Training / evaluation loop (shared by every method; the model file supplies the Summarizer)
#
# A Summarizer exposes:
#   sm.loss(batch, teacher=None)      -> scalar loss (teacher != None -> knowledge distillation)
#   sm.generate(batch, gen)           -> list of generated token-id lists
#   sm.pretrained_param_ids           -> ids of parameters that came from the Phase-1 checkpoint
# and the notebook supplies make_loader(split, cfg, shuffle, gen=False).
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def trainable_state(sm):
    return {n: p.detach().cpu().clone() for n, p in sm.named_parameters() if p.requires_grad}


def save_trainable(sm, path, extra=None):
    torch.save({"trainable": trainable_state(sm), **(extra or {})}, path)


def load_trainable(sm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in sm.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"checkpoint {path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]} "
                           f"(was it saved with the same method/config?)")
    sm.load_state_dict(ck["trainable"], strict=False)
    return ck


@torch.no_grad()
def eval_loss(sm, loader, max_batches=None):
    sm.eval(); tot, n = 0.0, 0
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with amp_ctx():
            tot += float(sm.loss(b)); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def generate_split(sm, split, cfg, n_docs=None, gen=None):
    """-> {id: summary text}"""
    sm.eval()
    gen = {**GEN, **(gen or {})}
    loader = make_loader(split, cfg, shuffle=False, gen=True, limit=n_docs)
    out, t0 = {}, time.time()
    for b in loader:
        with amp_ctx():
            seqs = sm.generate(b, gen)
        for rid, s in zip(b["ids"], seqs):
            out[rid] = postprocess(detok(s), gen.get("max_words"))
        if len(out) % (len(b["ids"]) * 10) == 0:
            print(f"    generated {len(out)} ({time.time() - t0:.0f}s)")
    return out


def score_split(preds, recs, bert=True):
    refs = {r["id"]: r["summ"] for r in recs}
    ids = [i for i in preds if i in refs]
    return evaluate_texts([preds[i] for i in ids], [refs[i] for i in ids], bert=bert)


def train_summarizer(sm, cfg, tag, teacher=None):
    tr, va = make_loader("train", cfg, shuffle=True), make_loader("val", cfg, shuffle=False)
    pre = sm.pretrained_param_ids
    new_p = [p for p in sm.parameters() if p.requires_grad and id(p) not in pre]
    old_p = [p for p in sm.parameters() if p.requires_grad and id(p) in pre]
    groups = ([dict(params=new_p, lr=cfg.lr)] if new_p else []) + ([dict(params=old_p, lr=cfg.lr_backbone)] if old_p else [])
    if not groups:
        raise RuntimeError("nothing to train")
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd, betas=(0.9, 0.98))
    total = max(1, cfg.epochs * math.ceil(len(tr) / cfg.accum))
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    use_scaler = AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    best, bad, step = -1.0, 0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    if teacher is not None:
        teacher.eval(); freeze(teacher)
    print(f"[{tag}] {len(tr)} batches/epoch x {cfg.epochs} epochs | accum {cfg.accum} | "
          f"trainable new {sum(p.numel() for p in new_p)/1e6:.2f}M + backbone {sum(p.numel() for p in old_p)/1e6:.2f}M")
    for ep in range(cfg.epochs):
        sm.train(); opt.zero_grad(set_to_none=True)
        t0, run = time.time(), 0.0
        for i, b in enumerate(tr):
            with amp_ctx():
                loss = sm.loss(b, teacher=teacher)
            if not torch.isfinite(loss):
                print(f"[{tag}] non-finite loss at batch {i}, skipped"); opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss / cfg.accum).backward()
            run += float(loss)
            if (i + 1) % cfg.accum == 0 or i + 1 == len(tr):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if (i + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} batch {i+1}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)")
        vl = eval_loss(sm, va, max_batches=cfg.val_loss_batches)
        preds = generate_split(sm, "val", cfg, n_docs=cfg.n_eval_gen)
        m = score_split(preds, val_recs, bert=cfg.select_metric == "bertscore")
        score = m[cfg.select_metric] if m.get(cfg.select_metric) is not None else m["rougeL"]
        print(f"[{tag}] ep {ep+1:02d} train {run/len(tr):.4f} | val loss {vl:.4f} | {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
        if score > best:
            best, bad = score, 0
            save_trainable(sm, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "val": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    load_trainable(sm, ck_path)
    shutil.copy(ck_path, os.path.join(p6_dir, f"ckpt_{tag}.pt"))          # persist the best checkpoint to Drive
    return best

## 1. Transformer-XL attention for summarisation (same weights, same maths) + KV cache
`memory` in a Transformer-XL block can be (a) `None`, (b) a tensor of earlier layer inputs (the Phase-1 segment memory, used here by prefix tuning),
or (c) a `KVCache`. With a cache, the grouped keys and values of every earlier position are stored, so each decoding step only projects the new token.
Positions are relative, so the positional term is simply recomputed for the current key length.

In [9]:
TXL_CTX = {"key_pad": None}            # [B, K] bool (True = real key) for the current call, set by txl_run()

if not hasattr(TransformerXLGQAAttention, "_orig_forward"):
    TransformerXLGQAAttention._orig_forward = TransformerXLGQAAttention.forward


class KVCache:
    """per-block cache: grouped keys / values [B, G, S, d_h]; `prefix` = learned memory [P, d] consumed on the first call"""

    def __init__(self, prefix=None):
        self.k = self.v = None
        self.prefix = prefix

    def reorder(self, idx):
        self.k, self.v = self.k.index_select(0, idx), self.v.index_select(0, idx)


def txl_attn_forward(self, x, memory=None, relative_position=None):
    B, Q, _ = x.shape
    cache = memory if isinstance(memory, KVCache) else None
    mem = None if cache is not None else memory
    if cache is not None and cache.k is None and cache.prefix is not None:
        mem = cache.prefix.unsqueeze(0).expand(B, -1, -1)              # prefix enters the cache on the first call
    kv_in = torch.cat([mem.to(x.dtype), x], 1) if mem is not None and mem.size(1) > 0 else x
    k = self._kv_heads(self.k_proj(kv_in))                              # [B, G, *, d_h]
    v = self._kv_heads(self.v_proj(kv_in))
    if cache is not None:
        if cache.k is not None:
            k, v = torch.cat([cache.k, k.to(cache.k.dtype)], 2), torch.cat([cache.v, v.to(cache.v.dtype)], 2)
        cache.k, cache.v = k, v
    K = k.size(2)
    q = self._q_heads(self.q_proj(x))
    k, v = self._expand_kv(k.to(q.dtype)), self._expand_kv(v.to(q.dtype))
    assert relative_position.size(0) == K, f"relative positions for {relative_position.size(0)} keys, have {K}"
    r = self._expand_kv(self._kv_heads(self.r_proj(relative_position.to(x.dtype)).unsqueeze(0)))      # [1,H,K,d_h]
    qv = q + self.v.to(q.dtype)[None, :, None, :]
    bd = self._rel_shift(torch.einsum("bhqd,hkd->bhqk", qv, r[0]))
    scale = 1.0 / math.sqrt(self.head_dim)
    blocked = self._causal_mask(Q, K, x.device).view(1, 1, Q, K)        # query i sees keys <= K-Q+i
    kp = TXL_CTX["key_pad"]
    if kp is not None:
        assert kp.shape == (B, K), f"key_pad {tuple(kp.shape)} != {(B, K)}"
        qpos = torch.arange(K - Q, K, device=x.device)[:, None]
        self_key = (torch.arange(K, device=x.device)[None, :] == qpos).view(1, 1, Q, K)
        blocked = (blocked | ~kp[:, None, None, :]) & ~self_key          # padded keys hidden; self always visible
    bias = (bd * scale).masked_fill(blocked, float("-inf")).to(q.dtype)
    qu = q + self.u.to(q.dtype)[None, :, None, :]
    drop = self.dropout.p if self.training else 0.0
    out = F.scaled_dot_product_attention(qu, k, v, attn_mask=bias, dropout_p=drop, scale=scale)
    return self.out_proj(out.transpose(1, 2).reshape(B, Q, self.d_model))


TransformerXLGQAAttention.forward = txl_attn_forward


def txl_run(g, ids=None, embeds=None, key_pad=None, caches=None, ckpt=False, upto=None):
    """Run the Transformer-XL stack -> final states [B,T,d] (after final_norm; parameter-free LayerNorm if upto is set).
    key_pad: [B, real positions so far] bool (prefix-memory columns are added here). caches: list of KVCache (generation).
    g.prefix (prefix tuning): learned memory [n_blocks, P, d]. Bottom blocks with no trainable parameter run under no_grad."""
    x = g.embedding(ids) if embeds is None else embeds
    B, Q = x.shape[:2]
    pre = getattr(g, "prefix", None)
    PL = 0 if pre is None else pre.size(1)
    first = caches is None or caches[0].k is None
    past = 0 if first else caches[0].k.size(2)                          # includes the prefix once it is cached
    K = past + Q + (PL if first else 0)
    rel = g.pos_encoding(Q, K, device=x.device)
    kp = None
    if key_pad is not None:
        kp = torch.cat([key_pad.new_ones(B, PL), key_pad], 1) if PL else key_pad
    blocks = g.blocks if upto is None else g.blocks[:upto + 1]
    lo = 0
    if torch.is_grad_enabled() and not x.requires_grad and PL == 0 and caches is None:
        while lo < len(blocks) and not any(p.requires_grad for p in blocks[lo].parameters()):
            lo += 1
    for i, blk in enumerate(blocks):
        if caches is not None:
            m = caches[i]
            if first and PL:
                m.prefix = pre[i]
        else:
            m = pre[i].unsqueeze(0).expand(B, -1, -1) if PL else None

        def run(h, blk=blk, m=m, kp=kp):
            TXL_CTX["key_pad"] = kp                                     # set again when checkpointing recomputes
            return blk(h, m, rel)

        if i < lo:
            with torch.no_grad():
                x = run(x)
        elif ckpt and torch.is_grad_enabled() and caches is None:
            x = grad_ckpt(run, x, use_reentrant=False)
        else:
            x = run(x)
    TXL_CTX["key_pad"] = None
    return g.final_norm(x) if upto is None else F.layer_norm(x, (x.size(-1),))


def new_caches(g):
    return [KVCache() for _ in g.blocks]


def reorder_cache(caches, idx):
    for c in caches:
        c.reorder(idx)
    return caches

## 1b. Self-test (seconds): run before any training

In [10]:
@torch.no_grad()
def _original_hidden(m, ids, memory=None):
    """the Phase-1 forward, block by block (memory = list of per-block tensors or None)"""
    TransformerXLGQAAttention.forward = TransformerXLGQAAttention._orig_forward
    try:
        x = m.embedding(ids)
        ml = 0 if memory is None else memory[0].size(1)
        rel = m.pos_encoding(ids.size(1), ml + ids.size(1), device=x.device)
        for i, blk in enumerate(m.blocks):
            x = blk(x, None if memory is None else memory[i], rel)
        return m.final_norm(x)
    finally:
        TransformerXLGQAAttention.forward = txl_attn_forward


@torch.no_grad()
def check_txl_patch():
    global AMP_DTYPE
    m = TXL_BASE.to(device).eval()
    docs = [d for d in tr_doc if len(d) >= 300][:4]
    ids = torch.as_tensor(np.stack([d[:256] for d in docs]), dtype=torch.long, device=device)
    tol = lambda ref: 1e-3 * max(1.0, ref.abs().max().item())

    # (1) new attention == Phase-1 attention, without and with Transformer-XL memory (prefix-tuning path)
    ref = _original_hidden(m, ids)
    new = txl_run(m, ids)
    d1 = (ref - new).abs().max().item()
    mem = torch.randn(len(m.blocks), 16, m.d_model, device=device)
    ref_m = _original_hidden(m, ids, [mem[i].unsqueeze(0).expand(ids.size(0), -1, -1) for i in range(len(m.blocks))])
    m.prefix = mem
    new_m = txl_run(m, ids)
    m.prefix = None
    d1m = (ref_m - new_m).abs().max().item()
    print(f"(1) patched vs Phase-1 attention        max|diff| {d1:.2e} | with 16 memory slots {d1m:.2e}")
    assert d1 < tol(ref) and d1m < tol(ref_m), "patched attention does not match the Phase-1 attention"

    # (2) left-padded prompt + cached incremental decoding == one full forward
    B, Tp, Tg = ids.size(0), 64, 24
    pad = [0, 7, 13, 30][:B]
    full = torch.full((B, Tp + Tg), PAD_ID, dtype=torch.long, device=device)
    kp = torch.zeros(B, Tp + Tg, dtype=torch.bool, device=device)
    for b in range(B):
        full[b, pad[b]:] = ids[b, : Tp + Tg - pad[b]]; kp[b, pad[b]:] = True
    h_full = txl_run(m, full, key_pad=kp)
    caches = new_caches(m)
    h_p = txl_run(m, full[:, :Tp], key_pad=kp[:, :Tp], caches=caches)
    steps = [h_p[:, -1]]
    for t in range(Tp, Tp + Tg - 1):
        steps.append(txl_run(m, full[:, t:t + 1], key_pad=kp[:, :t + 1], caches=caches)[:, -1])
    d2 = (torch.stack(steps, 1) - h_full[:, Tp - 1:Tp + Tg - 1]).abs().max().item()
    print(f"(2) cached decoding vs full forward     max|diff| {d2:.2e}")
    assert d2 < tol(h_full), "KV-cache decoding diverges from the full forward"

    # (3) left padding does not change the real positions
    solo = txl_run(m, full[B - 1:, pad[-1]:])
    d3 = (solo[0] - h_full[B - 1, pad[-1]:]).abs().max().item()
    print(f"(3) left-padded row vs unpadded row     max|diff| {d3:.2e} | NaN {bool(torch.isnan(h_full).any())}")
    assert d3 < tol(solo) and not torch.isnan(h_full).any()

    # (4) cached decoding WITH a learned prefix == full forward with the prefix
    m.prefix = mem
    h_full = txl_run(m, full, key_pad=kp)
    caches = new_caches(m)
    steps = [txl_run(m, full[:, :Tp], key_pad=kp[:, :Tp], caches=caches)[:, -1]]
    for t in range(Tp, Tp + 8):
        steps.append(txl_run(m, full[:, t:t + 1], key_pad=kp[:, :t + 1], caches=caches)[:, -1])
    m.prefix = None
    d4 = (torch.stack(steps, 1) - h_full[:, Tp - 1:Tp + 8]).abs().max().item()
    print(f"(4) cached decoding with prefix memory  max|diff| {d4:.2e}")
    assert d4 < tol(h_full)

    # (5) mixed precision closeness (switched off if it is not finite and close)
    if AMP_DTYPE is not None:
        with amp_ctx():
            h16 = txl_run(m, ids)
        ok = bool(torch.isfinite(h16).all())
        cos = F.cosine_similarity(h16.float().reshape(-1, h16.size(-1)), new.reshape(-1, new.size(-1)), -1).min().item() if ok else 0.0
        print(f"(5) {AMP_DTYPE} autocast vs fp32: finite {ok}, min cosine {cos:.5f}")
        if not ok or cos < 0.98:
            AMP_DTYPE = None; print("-> mixed precision switched OFF")
    print("Transformer-XL self-test passed")


check_txl_patch()

(1) patched vs Phase-1 attention        max|diff| 1.07e-06 | with 16 memory slots 1.07e-06
(2) cached decoding vs full forward     max|diff| 1.79e-06
(3) left-padded row vs unpadded row     max|diff| 8.34e-07 | NaN False
(4) cached decoding with prefix memory  max|diff| 2.26e-06
(5) torch.float16 autocast vs fp32: finite True, min cosine 1.00000
Transformer-XL self-test passed


## 2. Document memory (the "encoder" side of the summariser)
The frozen Phase-1 Transformer-XL reads every chunk once. Each chunk becomes m slot vectors: the means of its hidden states over m consecutive segments.
A causal state already summarises everything before it, so the later segments carry more of the chunk.

In [11]:
CHUNK_BODY = TXL_CTX_LEN - 2                      # <s> + 510 + </s>


@torch.no_grad()
def extract_memory(docs, K, m, strategy, layer=None, tok_budget=24576, max_bs=128, log_every=50):
    bb = TXL_BASE.to(device).eval()
    per_doc = [select_chunks(chunk_ids(d, CHUNK_BODY, CHUNK_BODY), K, strategy) for d in docs]
    flat = [c for cs in per_doc for c in cs]
    n, d = len(flat), bb.d_model
    lens = np.fromiter((len(c) + 2 for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")
    out = torch.empty(n, m, d, dtype=torch.float16)
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens -> {m} slots per chunk")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            k = lens[i] - 2
            arr[j, 0] = BOS_ID; arr[j, 1:k + 1] = flat[i]; arr[j, k + 1] = EOS_ID
        ids = torch.from_numpy(arr).to(device)
        ln = torch.from_numpy(lens[sel]).to(device)
        with amp_ctx():
            h = txl_run(bb, ids, upto=layer)                                  # right padding is safe (causal)
        h = h.float()
        pos = torch.arange(T, device=device)[None, :]
        seg = (pos * m // ln[:, None]).clamp(max=m - 1)
        oh = F.one_hot(seg, m).float() * (pos < ln[:, None]).unsqueeze(-1).float()
        out[torch.from_numpy(sel)] = (torch.einsum("btm,btd->bmd", oh, h) / oh.sum(1).clamp_min(1).unsqueeze(-1)).half().cpu()

    def safe_run(sel):
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache(); safe_run(sel[: len(sel) // 2]); safe_run(sel[len(sel) // 2:])

    t0, s, nb = time.time(), 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        safe_run(order[s:e]); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            print(f"    {s:,}/{n:,} chunks ({time.time() - t0:.0f}s)")
    feats, p = [], 0
    for cs in per_doc:
        feats.append(out[p:p + len(cs)].reshape(-1, d)); p += len(cs)
    return feats


def get_memory(split, docs, cfg):
    name = f"txlmem_{split}_s{SPLIT_SEED}_K{cfg.mem_chunks}_m{cfg.mem_slots}_{cfg.chunk_strategy}_L{cfg.mem_layer}.pt"
    dp, lp = os.path.join(p6_dir, name), os.path.join(FEAT_LOCAL, name)
    if not os.path.exists(lp) and os.path.exists(dp):
        shutil.copy(dp, lp)
    if os.path.exists(lp):
        return torch.load(lp, weights_only=False)
    print(f"extracting Transformer-XL memory for '{split}' ...")
    feats = extract_memory(docs, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy, None if cfg.mem_layer < 0 else cfg.mem_layer)
    torch.save(feats, lp); shutil.copy(lp, dp)
    return feats


class MemoryBridge(nn.Module):
    """slots [B,N,d] (left-padded) -> soft tokens in Transformer-XL's input-embedding space.
    Adds chunk / slot position embeddings, mixes the slots bidirectionally (the causal Transformer-XL features of one chunk
    never saw the other chunks), then projects and rescales to the token-embedding norm."""

    def __init__(self, d, n_chunks, n_slots, emb_std, layers=1, heads=8, dropout=0.1):
        super().__init__()
        self.n_slots = n_slots
        self.chunk_pos, self.slot_pos = nn.Embedding(n_chunks, d), nn.Embedding(n_slots, d)
        nn.init.normal_(self.chunk_pos.weight, std=0.02); nn.init.normal_(self.slot_pos.weight, std=0.02)
        self.ln_in = nn.LayerNorm(d)
        self.mix = (nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu",
                                                                     batch_first=True, norm_first=True),
                                          layers, enable_nested_tensor=False) if layers > 0 else None)
        self.ln_out, self.proj = nn.LayerNorm(d), nn.Linear(d, d)
        self.gain = nn.Parameter(torch.tensor(float(emb_std)))

    def forward(self, feats, mask, slot_idx):                 # slot_idx [B,N] = index of the slot inside its doc
        z = self.ln_in(feats) + self.chunk_pos(slot_idx // self.n_slots) + self.slot_pos(slot_idx % self.n_slots)
        if self.mix is not None:
            z = self.mix(z, src_key_padding_mask=~mask)
        return self.proj(self.ln_out(z)) * self.gain

## 3. Transformer-XL summariser + adaptation methods

In [12]:
TXL_LINEARS = ("attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.out_proj", "ffn.0", "ffn.3")


def _is_target(name):
    return name.startswith("blocks.") and any(name.endswith(t) for t in TXL_LINEARS)


@torch.no_grad()
def real_memory(g, P):
    """block inputs at P positions of a real training chunk: prefix-memory init at the right scale [n_blocks, P, d]"""
    doc = next(d for d in tr_doc if len(d) >= 300)
    ids = torch.as_tensor(np.concatenate([[BOS_ID], doc[:CHUNK_BODY], [EOS_ID]]), dtype=torch.long, device=device)[None]
    g = g.to(device)
    x = g.embedding(ids); rel = g.pos_encoding(ids.size(1), ids.size(1), device=device)
    ins = []
    for blk in g.blocks:
        ins.append(x); x = blk(x, None, rel)
    pos = torch.randperm(ids.size(1), device=device)[:P].sort().values
    return torch.stack([h[0, pos] for h in ins]).float()


def apply_backbone_method(g, cfg):
    """Sets requires_grad / injects modules into the Transformer-XL g. Returns a short description."""
    meth = cfg.method
    freeze(g)
    d = g.d_model
    if meth in ("full", "kd"):
        for p in g.parameters():
            p.requires_grad = True
        return "all Transformer-XL weights"
    if meth == "partial":
        k = cfg.unfreeze_top_k
        for blk in g.blocks[len(g.blocks) - k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in g.final_norm.parameters():
            p.requires_grad = True
        return f"top {k} blocks + final norm"
    if meth == "frozen":
        return "backbone frozen (memory bridge only)"
    if meth == "lora":
        n = replace_linears(g, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "lorma":
        n = replace_linears(g, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(g, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth in ("houlsby", "pfeiffer"):
        for blk in g.blocks:
            if meth == "houlsby":
                blk.attn = AdapterAfter(blk.attn, d, cfg.adapter_dim, cfg.adapter_dropout)
            blk.ffn = AdapterAfter(blk.ffn, d, cfg.adapter_dim, cfg.adapter_dropout)
        return f"{meth} adapters (bottleneck {cfg.adapter_dim})"
    if meth == "prefix":
        g.prefix = nn.Parameter(real_memory(g, cfg.prefix_len).cpu())
        return f"prefix tuning: {cfg.prefix_len} learned Transformer-XL memory slots per block"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens"
    raise ValueError(f"unknown method {meth!r}")


class TXLSummarizer(nn.Module):
    def __init__(self, g, cfg):
        super().__init__()
        self.g, self.cfg = g, cfg
        if not hasattr(g, "prefix"):
            g.prefix = None
        self.pretrained_param_ids = {id(p) for p in g.parameters()}            # before any injection
        self.method_desc = apply_backbone_method(g, cfg)
        d = g.d_model
        emb_std = float(g.embedding.weight.detach().float().std())
        self.bridge = (MemoryBridge(d, cfg.mem_chunks, cfg.mem_slots, emb_std, cfg.bridge_layers)
                       if cfg.mem_mode == "memory" else None)
        self.prompt = None
        if cfg.method == "prompt":
            self.prompt = nn.Parameter(g.embedding.weight.detach()[torch.randint(1000, 20000, (cfg.prompt_len,))].clone())

    def lm_logits(self, h):
        return F.linear(h, self.g.embedding.weight.to(h.dtype))                # tied LM head of Phase 1

    # ---- inputs: [prompt | memory (left-padded) | <s>] ----
    def _prefix_inputs(self, b):
        emb = self.g.embedding
        if self.bridge is not None:
            mask = b["mem_mask"].to(device)
            mem = self.bridge(b["mem"].to(device).float(), mask, b["slot_idx"].to(device))
        else:
            mask = b["lead_mask"].to(device)
            mem = emb(b["lead"].to(device))
        B = mem.size(0)
        dt = emb.weight.dtype
        parts, kps = [mem.to(dt)], [mask]
        if self.prompt is not None:
            parts.insert(0, self.prompt.to(dt).unsqueeze(0).expand(B, -1, -1))
            kps.insert(0, mask.new_ones(B, self.prompt.size(0)))
        parts.append(emb(torch.full((B, 1), BOS_ID, dtype=torch.long, device=device)))
        kps.append(mask.new_ones(B, 1))
        return torch.cat(parts, 1), torch.cat(kps, 1)

    def logits_targets(self, b):
        pe, pk = self._prefix_inputs(b)
        y, ym, tgt = b["y"].to(device), b["y_mask"].to(device), b["tgt"].to(device)
        x = torch.cat([pe, self.g.embedding(y).to(pe.dtype)], 1)
        h = txl_run(self.g, embeds=x, key_pad=torch.cat([pk, ym], 1), ckpt=self.cfg.grad_ckpt)
        Tp, L = pe.size(1), y.size(1)
        hs = h[:, Tp - 1:Tp + L]                                  # <s> predicts y_1 ... the last position predicts </s>
        sel = tgt != -100
        return self.lm_logits(hs[sel]), tgt[sel]

    def loss(self, b, teacher=None):
        logits, t = self.logits_targets(b)
        ce = F.cross_entropy(logits.float(), t, label_smoothing=self.cfg.label_smoothing)
        if teacher is None:
            return ce
        with torch.no_grad():
            tl, tt = teacher.logits_targets(b)
        assert tt.shape == t.shape
        T = self.cfg.kd_temp
        kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.log_softmax(tl.float() / T, -1),
                      log_target=True, reduction="batchmean") * (T * T)
        return (1 - self.cfg.kd_alpha) * ce + self.cfg.kd_alpha * kl

    @torch.no_grad()
    def generate(self, b, gen):
        g = self.g
        pe, pk = self._prefix_inputs(b)
        room = TXL_CTX_LEN - pe.size(1) - 1
        max_new = min(gen["max_new"], room) if gen.get("respect_ctx", True) else gen["max_new"]

        def start():
            caches = new_caches(g)
            h = txl_run(g, embeds=pe, key_pad=pk, caches=caches)
            return {"caches": caches, "kp": pk}, self.lm_logits(h[:, -1]).float()

        def step(st, tok):
            kp = torch.cat([st["kp"], st["kp"].new_ones(tok.size(0), 1)], 1)
            h = txl_run(g, ids=tok[:, None], key_pad=kp, caches=st["caches"])
            return {"caches": st["caches"], "kp": kp}, self.lm_logits(h[:, -1]).float()

        def reorder(st, idx):
            return {"caches": reorder_cache(st["caches"], idx), "kp": st["kp"].index_select(0, idx)}

        return beam_search(start, step, reorder, num_beams=gen["num_beams"], max_new=max_new,
                           min_new=min(gen["min_new"], max_new // 2), len_penalty=gen["len_penalty"],
                           no_repeat_ngram=gen["no_repeat_ngram"], rep_penalty=gen["rep_penalty"],
                           banned=(PAD_ID, BOS_ID))


def build_student_backbone(teacher_g, n_layers):
    """layer-strided student from the FINE-TUNED teacher (8 -> 4 keeps blocks [0, 2, 5, 7]), as in your TXL Phase-3/4/7;
    embedding (= tied LM head), positional encoding and final norm are copied."""
    keep = np.linspace(0, len(teacher_g.blocks) - 1, n_layers).round().astype(int).tolist()
    s = copy.deepcopy(teacher_g)
    s.blocks = nn.ModuleList([s.blocks[i] for i in keep])
    s.num_layers, s.prefix = n_layers, None
    print(f"student: {n_layers} blocks from teacher blocks {keep} | {sum(p.numel() for p in s.parameters())/1e6:.1f}M params "
          f"(teacher {sum(p.numel() for p in teacher_g.parameters())/1e6:.1f}M)")
    return s


INPUT_KEYS = ("mem_mode", "mem_chunks", "mem_slots", "chunk_strategy", "mem_layer", "bridge_layers", "max_sum_tok", "lead_tok")


def build_summarizer(cfg, backbone=None):
    g = copy.deepcopy(TXL_BASE) if backbone is None else backbone
    return TXLSummarizer(g, cfg).to(device)


def make_student_summarizer(teacher, cfg):
    sm = build_summarizer(cfg, backbone=build_student_backbone(teacher.g, cfg.student_layers))
    if teacher.bridge is not None:
        sm.bridge.load_state_dict(teacher.bridge.state_dict())         # start from the teacher's trained bridge
    return sm


@torch.no_grad()
def check_peft_identity():
    """LoRA / LoRMA+ / adapters must leave the backbone unchanged at initialisation; QLoRA stays close (4-bit base)."""
    ids = torch.as_tensor(np.stack([d[:128] for d in tr_doc if len(d) >= 128][:2]), dtype=torch.long, device=device)
    ref = txl_run(TXL_BASE.to(device).eval(), ids)
    for meth in ("lora", "lorma", "houlsby", "pfeiffer", "qlora"):
        g = copy.deepcopy(TXL_BASE); g.prefix = None
        apply_backbone_method(g, SimpleNamespace(**{**BASE_CFG, "method": meth}))
        g = g.to(device).eval()
        out = txl_run(g, ids)
        d = (out - ref).abs().max().item()
        if meth == "qlora":
            cos = F.cosine_similarity(out.reshape(-1, out.size(-1)), ref.reshape(-1, ref.size(-1)), -1).mean().item()
            print(f"  qlora     4-bit base: mean cosine to fp32 {cos:.4f}")
        else:
            print(f"  {meth:9s} identity at init: max|diff| {d:.2e}" + ("" if d < 1e-4 else "   <-- NOT identity!"))
            assert d < 1e-4, f"{meth} is not the identity at initialisation"
        del g
    torch.cuda.empty_cache()

## 4. Datasets / collate (identical to the GQLA notebook)

In [13]:
# ============================================================
# 4. DATASETS / COLLATE
# ============================================================
SPLITS = {"train": (train_recs, tr_doc, tr_sum), "val": (val_recs, va_doc, va_sum), "test": (test_recs, te_doc, None)}
_DS = {}


class SummDS(Dataset):
    def __init__(self, split, cfg):
        recs, docs, sums = SPLITS[split]
        self.ids = [r["id"] for r in recs]
        self.src = (get_memory(split, docs, cfg) if cfg.mem_mode == "memory"
                    else [np.asarray(d[:cfg.lead_tok], dtype=np.int64) for d in docs])
        self.sums = sums

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.src[i], (None if self.sums is None else self.sums[i])


def make_collate(cfg, with_target):
    max_y = cfg.max_sum_tok

    def collate(batch):
        B = len(batch)
        out = {"ids": [b[0] for b in batch]}
        if cfg.mem_mode == "memory":
            N = max(b[1].size(0) for b in batch); d = batch[0][1].size(1)
            mem = torch.zeros(B, N, d, dtype=torch.float16)
            mask = torch.zeros(B, N, dtype=torch.bool); slot = torch.zeros(B, N, dtype=torch.long)
            for i, (_, f, _) in enumerate(batch):                     # LEFT-pad: the memory ends right before <s>
                n = f.size(0)
                mem[i, N - n:] = f; mask[i, N - n:] = True; slot[i, N - n:] = torch.arange(n)
            out.update(mem=mem, mem_mask=mask, slot_idx=slot)
        else:
            N = max(max(len(b[1]), 1) for b in batch)
            lead = torch.full((B, N), PAD_ID, dtype=torch.long); mask = torch.zeros(B, N, dtype=torch.bool)
            for i, (_, t, _) in enumerate(batch):
                if len(t):
                    lead[i, N - len(t):] = torch.from_numpy(t); mask[i, N - len(t):] = True
                else:
                    mask[i, -1] = True
            out.update(lead=lead, lead_mask=mask)
        if with_target:
            ys = [np.asarray(b[2][:max_y], dtype=np.int64) for b in batch]
            L = max(max(len(y) for y in ys), 1)
            y = torch.full((B, L), PAD_ID, dtype=torch.long); ym = torch.zeros(B, L, dtype=torch.bool)
            tgt = torch.full((B, L + 1), -100, dtype=torch.long)
            for i, s in enumerate(ys):
                n = len(s)
                y[i, :n] = torch.from_numpy(s); ym[i, :n] = True
                tgt[i, :n] = torch.from_numpy(s); tgt[i, n] = EOS_ID
            out.update(y=y, y_mask=ym, tgt=tgt)
        return out
    return collate


def make_loader(split, cfg, shuffle, gen=False, limit=None):
    key = (split, cfg.mem_mode, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy, cfg.mem_layer, cfg.lead_tok)
    if key not in _DS:
        _DS[key] = SummDS(split, cfg)
    ds = _DS[key]
    if limit:
        ds = torch.utils.data.Subset(ds, list(range(min(limit, len(ds)))))
    elif split == "train" and cfg.max_train_docs:
        ds = torch.utils.data.Subset(ds, list(range(min(cfg.max_train_docs, len(ds)))))
    has_y = split != "test" and not gen
    return DataLoader(ds, batch_size=cfg.gen_bs if gen else cfg.batch_size, shuffle=shuffle,
                      collate_fn=make_collate(cfg, has_y), num_workers=0, drop_last=False)

## 5. Configuration

In [14]:
# ============================================================
# 5. CONFIGURATION
# ============================================================
BASE_CFG = dict(
    method="lora",
    # document memory: K chunks x m slots soft tokens (16 x 8 = 128 of the 512 positions)
    mem_mode="memory",            # "memory" (bridge over cached Transformer-XL states) | "lead" (first tokens of the doc as text)
    mem_chunks=16, mem_slots=8, chunk_strategy="uniform", mem_layer=-1, bridge_layers=1,
    lead_tok=None,                # lead mode: filled in by make_cfg to use the whole remaining window
    max_sum_tok=None,             # filled in by make_cfg: min(p95 of reference lengths, room left in 512)
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, adapter_dim=64, adapter_dropout=0.1,
    prefix_len=16, prompt_len=20, unfreeze_top_k=3,
    # optimisation
    lr=5e-4, lr_backbone=5e-5, wd=0.01, epochs=6, patience=2, batch_size=8, accum=2, warmup=0.05, clip=1.0,
    label_smoothing=0.1, grad_ckpt=True, seed=42, log_every=100, max_train_docs=None,
    # model selection during training (generation on a slice of val is the honest signal, not loss)
    val_loss_batches=20, n_eval_gen=64, select_metric="rougeL", gen_bs=8, n_eval_final=None,
    # knowledge distillation
    teacher_ckpt=None, student_layers=4, kd_alpha=0.5, kd_temp=2.0,
)

METHOD_DEFAULTS = {                                    # sensible starting points per method
    "full":     dict(lr=5e-4, lr_backbone=3e-5),
    "partial":  dict(lr=5e-4, lr_backbone=5e-5),
    "frozen":   dict(lr=1e-3),
    "lora":     dict(lr=5e-4),
    "lorma":    dict(lr=5e-4),
    "qlora":    dict(lr=5e-4),
    "houlsby":  dict(lr=5e-4),
    "pfeiffer": dict(lr=5e-4),
    "prefix":   dict(lr=2e-3),
    "prompt":   dict(lr=3e-3),
    "kd":       dict(lr=5e-4, lr_backbone=5e-5),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    c = SimpleNamespace(**{**BASE_CFG, **METHOD_DEFAULTS.get(meth, {}), **over})
    P = c.prompt_len if c.method == "prompt" else 0
    if c.mem_mode == "memory":
        n_mem = c.mem_chunks * c.mem_slots
        room = TXL_CTX_LEN - P - n_mem - 1 - 1                # -1 for <s>, -1 for </s>
        c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P95, room)
        assert c.max_sum_tok <= room, f"summary budget {c.max_sum_tok} > room {room}"
    else:
        c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P90, TXL_CTX_LEN // 2)
        c.lead_tok = c.lead_tok or TXL_CTX_LEN - P - 1 - c.max_sum_tok - 1
    return c


GEN = dict(num_beams=4, max_new=SUM_TOK_P90, min_new=SUM_TOK_P10, len_penalty=1.0,
           no_repeat_ngram=3, rep_penalty=1.0, max_words=None, respect_ctx=True)

_c = make_cfg()
print(f"memory layout: {_c.mem_chunks} chunks x {_c.mem_slots} slots = {_c.mem_chunks * _c.mem_slots} soft tokens "
      f"| summary budget {_c.max_sum_tok} tokens (reference p95 = {SUM_TOK_P95})")
_trunc = (sl > _c.max_sum_tok).mean()
if _trunc > 0.05:
    print(f"NOTE: {_trunc:.0%} of training summaries are longer than the {_c.max_sum_tok}-token budget and are "
          f"truncated; the 512-token window is the limit here. Fewer memory slots (mem_chunks / mem_slots) "
          f"buy more summary room.")

check_peft_identity()

memory layout: 16 chunks x 8 slots = 128 soft tokens | summary budget 382 tokens (reference p95 = 2399)
NOTE: 86% of training summaries are longer than the 382-token budget and are truncated; the 512-token window is the limit here. Fewer memory slots (mem_chunks / mem_slots) buy more summary room.
  lora      identity at init: max|diff| 0.00e+00
  lorma     identity at init: max|diff| 7.15e-07
  houlsby   identity at init: max|diff| 0.00e+00
  pfeiffer  identity at init: max|diff| 0.00e+00
  qlora     4-bit base: mean cosine to fp32 0.9999


## 6. Experiment runner (identical to the GQLA notebook)

In [15]:
# ============================================================
# 6. EXPERIMENT RUNNER
# ============================================================
RESULTS = {}
_res_path = os.path.join(p6_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def _teacher_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    tcfg = SimpleNamespace(**ck["cfg"])
    t = build_summarizer(tcfg)
    load_trainable(t, path)
    return t, tcfg


def evaluate_and_submit(sm, cfg, name, submit=True):
    t0 = time.time()
    val_pred = generate_split(sm, "val", cfg, n_docs=getattr(cfg, "n_eval_final", None))
    m = score_split(val_pred, val_recs, bert=True)
    print(f"[{name}] VAL ({len(val_pred)} docs): {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
    RESULTS[name] = dict(val=m, method=cfg.method, desc=sm.method_desc,
                         trainable=n_params(sm), total=n_params(sm, trainable=False),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    json.dump(val_pred, open(os.path.join(p6_dir, f"valpred_{name}.json"), "w"), ensure_ascii=False)
    if submit:
        test_pred = generate_split(sm, "test", cfg)
        write_submission(test_pred, name)
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    teacher = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. the full fine-tune)"
        teacher, tcfg = _teacher_from_ckpt(cfg.teacher_ckpt)
        for k in INPUT_KEYS:
            setattr(cfg, k, getattr(tcfg, k))                       # student reads exactly the teacher's inputs
        sm = make_student_summarizer(teacher, cfg)
    else:
        sm = build_summarizer(cfg)
    print(f"\n=== {name} | {sm.method_desc} | trainable {n_params(sm)/1e6:.2f}M / {n_params(sm, False)/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_summarizer(sm, cfg, name, teacher=teacher)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
    del teacher; gc.collect(); torch.cuda.empty_cache()
    m = evaluate_and_submit(sm, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del sm; gc.collect(); torch.cuda.empty_cache()
    return m


def resubmit(name, **gen_over):
    """Re-generate val + test from a saved checkpoint with different decoding (no retraining)."""
    path = os.path.join(p6_dir, f"ckpt_{name}.pt")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        teacher, _ = _teacher_from_ckpt(cfg.teacher_ckpt)
        sm = make_student_summarizer(teacher, cfg); del teacher
    else:
        sm = build_summarizer(cfg)
    load_trainable(sm, path)
    saved = dict(GEN); GEN.update(gen_over)
    tag = name + "_" + "_".join(f"{k}{v}" for k, v in gen_over.items())
    try:
        return evaluate_and_submit(sm, cfg, tag)
    finally:
        GEN.clear(); GEN.update(saved); del sm; gc.collect(); torch.cuda.empty_cache()


def show_results():
    print(f"\n{'experiment':26s} {'method':9s} {'BERTScore':>9s} {'R1':>7s} {'R2':>7s} {'RL':>7s} {'words':>6s} "
          f"{'train M':>8s} {'total M':>8s} {'min':>6s} {'GB':>5s}")
    for n, r in RESULTS.items():
        v = r["val"]; bs = v.get("bertscore")
        print(f"{n:26s} {r['method']:9s} {('%.4f' % bs) if bs is not None else 'n/a':>9s} {v['rouge1']:7.4f} "
              f"{v['rouge2']:7.4f} {v['rougeL']:7.4f} {v['avg_words']:6.0f} {r['trainable']/1e6:8.2f} "
              f"{r['total']/1e6:8.1f} {r.get('train_min', 0):6.0f} {r.get('peak_gb', 0):5.1f}")

## 7. RUN
Run the smoke test first. The first run extracts and caches the memory for train/val/test, a one-off of roughly 10–20 min on a T4.
Then run the experiments one cell at a time: each one saves its checkpoint, validation predictions and `submission_<name>/submission.zip` to Drive.

In [ ]:
run_experiment("T0_smoke", method="lora", max_train_docs=64, epochs=1, n_eval_gen=8, n_eval_final=16, submit=False)
show_results()


=== T0_smoke | LoRA r=16 on 192 linears | trainable 7.75M / 114.9M ===
extracting Transformer-XL memory for 'train' ...
  57,444 chunks | 27,813,331 tokens -> 8 slots per chunk
    4,932/57,444 chunks (127s)
    7,493/57,444 chunks (369s)
    9,893/57,444 chunks (598s)
    12,293/57,444 chunks (827s)
    14,693/57,444 chunks (1056s)
    17,093/57,444 chunks (1285s)
    19,493/57,444 chunks (1514s)
    21,893/57,444 chunks (1743s)
    24,293/57,444 chunks (1972s)
    26,693/57,444 chunks (2201s)
    29,093/57,444 chunks (2429s)
    31,493/57,444 chunks (2658s)
    33,893/57,444 chunks (2887s)
    36,293/57,444 chunks (3116s)
    38,693/57,444 chunks (3345s)
    41,093/57,444 chunks (3574s)
    43,493/57,444 chunks (3804s)
    45,893/57,444 chunks (4032s)
    48,293/57,444 chunks (4261s)
    50,693/57,444 chunks (4490s)
    53,093/57,444 chunks (4719s)
    55,493/57,444 chunks (4949s)
    57,444/57,444 chunks (5135s)
extracting Transformer-XL memory for 'val' ...
  3,034 chunks | 1,470,

/tmp/ipykernel_1000/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  run += float(loss)


[T0_smoke] ep 01 train 7.4470 | val loss 7.3742 | BERTScore n/a | R1 0.2611 R2 0.0525 RL 0.1668 | words 424 (ref 501) (89s)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[T0_smoke] VAL (16 docs): BERTScore 0.7138 | R1 0.2732 R2 0.0550 RL 0.1746 | words 396 (ref 521) (132s)

experiment                 method    BERTScore      R1      R2      RL  words  train M  total M    min    GB
T0_smoke                   lora         0.7138  0.2732  0.0550  0.1746    396     7.75    114.9     92   5.6


In [ ]:
# Six substantially different methods (LoRMA compulsory, KD mandatory). Same memory input and decoding,
# so the differences come from the adaptation method. Run one per cell if Colab sessions are short.
run_experiment("T1_full",    method="full", batch_size=4, accum=4)        # also the KD teacher


=== T1_full | all Transformer-XL weights | trainable 110.57M / 110.6M ===
[T1_full] 1670 batches/epoch x 6 epochs | accum 4 | trainable new 3.43M + backbone 107.14M
[T1_full] ep 1 batch 100/1670 loss 7.4386 (162s)
[T1_full] ep 1 batch 200/1670 loss 7.3976 (321s)
[T1_full] ep 1 batch 300/1670 loss 7.3690 (482s)
[T1_full] ep 1 batch 400/1670 loss 7.3494 (642s)
[T1_full] ep 1 batch 500/1670 loss 7.3342 (802s)
[T1_full] ep 1 batch 600/1670 loss 7.3153 (962s)
[T1_full] ep 1 batch 700/1670 loss 7.2969 (1121s)
[T1_full] ep 1 batch 800/1670 loss 7.2851 (1279s)
[T1_full] ep 1 batch 900/1670 loss 7.2707 (1440s)
[T1_full] ep 1 batch 1000/1670 loss 7.2592 (1599s)
[T1_full] ep 1 batch 1100/1670 loss 7.2509 (1758s)
[T1_full] ep 1 batch 1200/1670 loss 7.2425 (1915s)
[T1_full] ep 1 batch 1300/1670 loss 7.2339 (2074s)
[T1_full] ep 1 batch 1400/1670 loss 7.2262 (2234s)
[T1_full] ep 1 batch 1500/1670 loss 7.2208 (2394s)
[T1_full] ep 1 batch 1600/1670 loss 7.2149 (2554s)
[T1_full] ep 01 train 7.2109 | va

{'rouge1': 0.2711057268217871,
 'rouge2': 0.05467045543382792,
 'rougeL': 0.20188868301756505,
 'bertscore': 0.72426438331604,
 'avg_words': 188.7094017094017,
 'ref_words': 824.957264957265}

In [ ]:
run_experiment("T2_lora",    method="lora")


=== T2_lora | LoRA r=16 on 192 linears | trainable 7.75M / 114.9M ===
[T2_lora] 835 batches/epoch x 6 epochs | accum 2 | trainable new 7.75M + backbone 0.00M
[T2_lora] ep 1 batch 100/835 loss 7.3921 (330s)
[T2_lora] ep 1 batch 200/835 loss 7.3873 (661s)
[T2_lora] ep 1 batch 300/835 loss 7.3752 (990s)
[T2_lora] ep 1 batch 400/835 loss 7.3652 (1317s)
[T2_lora] ep 1 batch 500/835 loss 7.3617 (1647s)
[T2_lora] ep 1 batch 600/835 loss 7.3615 (1979s)
[T2_lora] ep 1 batch 700/835 loss 7.3599 (2310s)
[T2_lora] ep 1 batch 800/835 loss 7.3588 (2640s)
[T2_lora] ep 01 train 7.3582 | val loss 7.3362 | BERTScore n/a | R1 0.2598 R2 0.0509 RL 0.1689 | words 384 (ref 620) (3076s)
[T2_lora] ep 2 batch 100/835 loss 7.3310 (330s)


In [16]:
run_experiment("T3_lorma",   method="lorma")                             # compulsory


=== T3_lorma | LoRMA+ r=16 on 192 linears | trainable 7.36M / 114.5M ===
[T3_lorma] 835 batches/epoch x 6 epochs | accum 2 | trainable new 7.36M + backbone 0.00M


/tmp/ipykernel_1985/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  run += float(loss)


[T3_lorma] ep 1 batch 100/835 loss 7.3773 (323s)
[T3_lorma] ep 1 batch 200/835 loss 7.3580 (652s)
[T3_lorma] ep 1 batch 300/835 loss 7.3591 (983s)
[T3_lorma] ep 1 batch 400/835 loss 7.3582 (1308s)
[T3_lorma] ep 1 batch 500/835 loss 7.3577 (1638s)
[T3_lorma] ep 1 batch 600/835 loss 7.3593 (1965s)
[T3_lorma] ep 1 batch 700/835 loss 7.3603 (2293s)
[T3_lorma] ep 1 batch 800/835 loss 7.3600 (2620s)
[T3_lorma] ep 01 train 7.3595 | val loss 7.3372 | BERTScore n/a | R1 0.2546 R2 0.0477 RL 0.1670 | words 385 (ref 620) (3062s)
[T3_lorma] ep 2 batch 100/835 loss 7.3627 (327s)
[T3_lorma] ep 2 batch 200/835 loss 7.3619 (657s)
[T3_lorma] ep 2 batch 300/835 loss 7.3595 (986s)
[T3_lorma] ep 2 batch 400/835 loss 7.3591 (1316s)
[T3_lorma] ep 2 batch 500/835 loss 7.3575 (1644s)
[T3_lorma] ep 2 batch 600/835 loss 7.3581 (1972s)
[T3_lorma] ep 2 batch 700/835 loss 7.3580 (2300s)
[T3_lorma] ep 2 batch 800/835 loss 7.3560 (2627s)
[T3_lorma] ep 02 train 7.3551 | val loss 7.3366 | BERTScore n/a | R1 0.2559 R2 0

{'rouge1': 0.2610787291818391,
 'rouge2': 0.046559932050488625,
 'rougeL': 0.17681529572338342,
 'bertscore': None,
 'avg_words': 375.06267806267806,
 'ref_words': 824.957264957265}

In [17]:
run_experiment("T4_houlsby", method="houlsby")


=== T4_houlsby | houlsby adapters (bottleneck 64) | trainable 7.66M / 114.8M ===
[T4_houlsby] 835 batches/epoch x 6 epochs | accum 2 | trainable new 7.66M + backbone 0.00M
[T4_houlsby] ep 1 batch 100/835 loss 7.3803 (316s)
[T4_houlsby] ep 1 batch 200/835 loss 7.3649 (636s)
[T4_houlsby] ep 1 batch 300/835 loss 7.3617 (954s)
[T4_houlsby] ep 1 batch 400/835 loss 7.3671 (1272s)
[T4_houlsby] ep 1 batch 500/835 loss 7.3617 (1590s)
[T4_houlsby] ep 1 batch 600/835 loss 7.3601 (1907s)
[T4_houlsby] ep 1 batch 700/835 loss 7.3582 (2223s)
[T4_houlsby] ep 1 batch 800/835 loss 7.3583 (2538s)
[T4_houlsby] ep 01 train 7.3594 | val loss 7.3352 | BERTScore n/a | R1 0.2544 R2 0.0481 RL 0.1677 | words 384 (ref 620) (2890s)
[T4_houlsby] ep 2 batch 100/835 loss 7.3423 (317s)
[T4_houlsby] ep 2 batch 200/835 loss 7.3497 (635s)
[T4_houlsby] ep 2 batch 300/835 loss 7.3497 (952s)
[T4_houlsby] ep 2 batch 400/835 loss 7.3502 (1266s)
[T4_houlsby] ep 2 batch 500/835 loss 7.3529 (1584s)
[T4_houlsby] ep 2 batch 600/8

{'rouge1': 0.2594205671423136,
 'rouge2': 0.045770440033525525,
 'rougeL': 0.17686724763165235,
 'bertscore': None,
 'avg_words': 375.5868945868946,
 'ref_words': 824.957264957265}

In [18]:
run_experiment("T5_prefix",  method="prefix")                            # learned Transformer-XL memory per block


=== T5_prefix | prefix tuning: 16 learned Transformer-XL memory slots per block | trainable 3.69M / 110.8M ===
[T5_prefix] 835 batches/epoch x 6 epochs | accum 2 | trainable new 3.69M + backbone 0.00M
[T5_prefix] ep 1 batch 100/835 loss 7.4584 (325s)
[T5_prefix] ep 1 batch 200/835 loss 7.4497 (654s)
[T5_prefix] ep 1 batch 300/835 loss 7.4475 (980s)
[T5_prefix] ep 1 batch 400/835 loss 7.4445 (1302s)
[T5_prefix] ep 1 batch 500/835 loss 7.4398 (1629s)
[T5_prefix] ep 1 batch 600/835 loss 7.4403 (1954s)
[T5_prefix] ep 1 batch 700/835 loss 7.4380 (2281s)
[T5_prefix] ep 1 batch 800/835 loss 7.4383 (2608s)
[T5_prefix] ep 01 train 7.4377 | val loss 7.4185 | BERTScore n/a | R1 0.2617 R2 0.0509 RL 0.1749 | words 354 (ref 620) (2937s)
[T5_prefix] ep 2 batch 100/835 loss 7.4401 (329s)
[T5_prefix] ep 2 batch 200/835 loss 7.4430 (659s)
[T5_prefix] ep 2 batch 300/835 loss 7.4333 (985s)
[T5_prefix] ep 2 batch 400/835 loss 7.4298 (1313s)
[T5_prefix] ep 2 batch 500/835 loss 7.4297 (1640s)
[T5_prefix] ep

{'rouge1': 0.2648711214269824,
 'rouge2': 0.05079555501204299,
 'rougeL': 0.1823651065951619,
 'bertscore': None,
 'avg_words': 343.16809116809117,
 'ref_words': 824.957264957265}

In [19]:
run_experiment("T6_kd", method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_T1_full.pt"), student_layers=4)   # blocks [0, 2, 5, 7]
show_results()

student: 4 blocks from teacher blocks [0, 10, 21, 31] | 28.1M params (teacher 107.1M)

=== T6_kd | all Transformer-XL weights | trainable 31.50M / 31.5M ===
[T6_kd] 835 batches/epoch x 6 epochs | accum 2 | trainable new 3.43M + backbone 28.07M


/tmp/ipykernel_1985/1471079892.py:104: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1


[T6_kd] ep 1 batch 100/835 loss 36.3429 (141s)
[T6_kd] ep 1 batch 200/835 loss 19.9865 (283s)
[T6_kd] ep 1 batch 300/835 loss 14.5075 (425s)
[T6_kd] ep 1 batch 400/835 loss 11.7495 (568s)
[T6_kd] ep 1 batch 500/835 loss 10.0793 (710s)
[T6_kd] ep 1 batch 600/835 loss 8.9592 (851s)
[T6_kd] ep 1 batch 700/835 loss 8.1553 (993s)
[T6_kd] ep 1 batch 800/835 loss 7.5513 (1135s)
[T6_kd] ep 01 train 7.3738 | val loss 6.4657 | BERTScore n/a | R1 0.3883 R2 0.1269 RL 0.2232 | words 313 (ref 620) (1222s)
[T6_kd] ep 2 batch 100/835 loss 3.3007 (142s)
[T6_kd] ep 2 batch 200/835 loss 3.2939 (284s)
[T6_kd] ep 2 batch 300/835 loss 3.2826 (425s)
[T6_kd] ep 2 batch 400/835 loss 3.2779 (568s)
[T6_kd] ep 2 batch 500/835 loss 3.2727 (710s)
[T6_kd] ep 2 batch 600/835 loss 3.2680 (851s)
[T6_kd] ep 2 batch 700/835 loss 3.2639 (993s)
[T6_kd] ep 2 batch 800/835 loss 3.2589 (1135s)
[T6_kd] ep 02 train 3.2579 | val loss 6.2692 | BERTScore n/a | R1 0.3887 R2 0.1305 RL 0.2198 | words 305 (ref 620) (1221s)
[T6_kd] ep 

In [ ]:
# ---- optional extras / ablations ----
# run_experiment("T7_qlora",    method="qlora")                            # 4-bit base: memory vs quality
# run_experiment("T8_pfeiffer", method="pfeiffer")
# run_experiment("T9_prompt",   method="prompt")
# run_experiment("T10_frozen",  method="frozen")                           # bridge only: how much must the backbone adapt?
# run_experiment("T11_partial", method="partial", unfreeze_top_k=3)
# run_experiment("T12_lead",    method="lora", mem_mode="lead")            # plain text prefix, no memory
# run_experiment("T13_headtail",method="lora", chunk_strategy="head_tail") # which parts of the judgment matter
# run_experiment("T14_m4",      method="lora", mem_slots=4)                # shorter memory, longer summaries
# Decoding sweeps reuse a trained checkpoint (no retraining):
# resubmit("T3_lorma", len_penalty=1.2, min_new=200)
# resubmit("T3_lorma", num_beams=1)
show_results()

### Report checklist (brief §12.5)
* **Technique comparison:** BERTScore / ROUGE against trainable parameters in `show_results()`, and how close PEFT gets to `T1_full`.
* **LoRA vs LoRMA⁺** at the same rank and targets (T2 vs T3): additive vs multiplicative updates.
* **Long documents:** which chunks feed the memory (`chunk_strategy`), memory size (`mem_slots`, `mem_chunks`), and the trade-off against summary room. Compare with `mem_mode="lead"` (T12).
* **Decoding:** beam size, length penalty, `min_new` (via `resubmit`), and summary length vs reference length.
* **KD:** T6 vs its teacher T1 in quality, size (4 vs 8 blocks), generation speed and memory.
* **Architecture:** compare with the GQLA and encoder–decoder Phase-6 results (same pipeline). Transformer-XL's segment recurrence is not used across chunks; state this.